In [277]:
class Value:
    def __init__(self,val,_children=(),_op=''):
        self.data = val
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda:None
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data,(self,other),'+')
        def add_backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = add_backward
        return out

    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other),'*')
        def mul_backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = mul_backward
        return out

    def __pow__(self,p):
        assert isinstance(p,(int,float))
        out = Value(self.data ** p , (self,),f'**{p}')
        def pow_backward():
            self.grad += out.grad * p * self.data ** (p - 1)
        out._backward = pow_backward
        return out
    
    def __truediv__(self,other):
        return self * other**-1

    def __neg__(self):
        return -1 * self
    
    def __sub__(self,other):
        return -other + self

    def __rsub__(self,other):
        return -self + other

    def __rtruediv__(self,other):
        return other * self**-1 
    
    def __radd__(self,other):
        return self.__add__(other)
    
    def __rmul__(self,other):
        return self.__mul__(other)

    def sin(self):
        import math
        out = Value(math.sin(self.data),(self,),'sin')
        def sin_backward():
            self.grad += out.grad * math.cos(self.data)
        
        out._backward = sin_backward
        return out

    def cos(self):
        import math
        out = Value(math.cos(self.data),(self,),'cos')
        def cos_backward():
            self.grad -= out.grad * math.sin(self.data)

        out._backward = cos_backward
        return out
    
    def tanh(self):
        val = self.data
        
        import math
        
        val = (math.exp(2*val) - 1)/(math.exp(2*val) + 1) 
        def tanh_backward():
            self.grad += out.grad * (1 - val**2)
        out =  Value(val,(self,),'tanh')
        out._backward = tanh_backward
        return out
   
    def exp(self):
        import math
        out = Value(math.exp(self.data),(self,),'exp')
        def exp_backward():
            self.grad += out.grad * out.data
        out._backward = exp_backward
        return out        
    
    def backward(self):
        visited = set()
        topo = []
        def recursion(root):
            if root not in visited:
                root.grad=0
                visited.add(root)
                for node in root._prev:
                    recursion(node)
                topo.append(root)
        recursion(self)
        topo.reverse()
        self.grad = 1
        for node in topo:
            node._backward()

In [356]:
def test_sanity_check():
    import torch
    x = Value(-4.0)
    z = 2 * x + 2 + x
    q = z.tanh() + z * x
    h = (z * z).tanh()
    y = h + q + q * x
    y.backward()
    xmg, ymg = x, y

    x = torch.Tensor([-4.0]).double()
    x.requires_grad = True
    z = 2 * x + 2 + x
    q = z.tanh() + z * x
    h = (z * z).tanh()
    y = h + q + q * x
    y.backward()
    xpt, ypt = x, y

    # forward pass went well
    assert ymg.data == ypt.data.item()
    # backward pass went well
    assert xmg.grad == xpt.grad.item()

def test_more_ops():
    import torch
    a = Value(-4.0)
    b = Value(2.0)
    c = a + b
    d = a * b + b**3
    c += c + 1
    c += 1 + c + (-a)
    d += d * 2 + (b + a).tanh()
    d += 3 * d + (b - a).tanh()
    e = c - d
    f = e**2
    g = f / 2.0
    g += 10.0 / f
    g.backward()
    amg, bmg, gmg = a, b, g

    a = torch.Tensor([-4.0]).double()
    b = torch.Tensor([2.0]).double()
    a.requires_grad = True
    b.requires_grad = True
    c = a + b
    d = a * b + b**3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).tanh()
    d = d + 3 * d + (b - a).tanh()
    e = c - d
    f = e**2
    g = f / 2.0
    g = g + 10.0 / f
    g.backward()
    apt, bpt, gpt = a, b, g

    tol = 1e-6
    # forward pass went well
    assert abs(gmg.data - gpt.data.item()) < tol
    # backward pass went well
    print(amg.grad,apt.grad.item())
    assert abs(amg.grad - apt.grad.item()) < tol
    assert abs(bmg.grad - bpt.grad.item()) < tol

In [357]:
test_sanity_check()

In [358]:
test_more_ops()

27.060135135849574 27.060135135849503


In [362]:
import random
class Neuron:
    def __init__(self,n):
        self.n = n
        self.w =[Value(random.uniform(-1,1)) for _ in range(n)]
        self.b = Value(random.uniform(-1,1))
    def forward(self,inp):
        out = sum((wi*ii for ii,wi in zip(inp,self.w)),self.b)
        out = out.tanh()
        return out
    def backward(self):
        self.w = [Value(wi.data - 0.2 * wi.grad) for wi in self.w]
        self.b = Value(self.b.data - 0.2 * self.b.grad)

class Layer:
    def __init__(self,pre=1,n=1):
        self.ns = [Neuron(pre) for i in range(n)]

    def forward(self,inp):
        out = [neu.forward(inp) for neu in self.ns]
        return out

    def backward(self,):
        for neu in self.ns:
            neu.backward()

class MLP:
        def __init__(self,input_num,layer_nums):
            self.loss = 0
            sz = [input_num]+layer_nums
            self.layers =  [Layer(sz[i],sz[i+1]) for i in range(len(layer_nums))]

        def __repr__(self):
            return f"loss : {self.loss}"
        
        def forward(self,inp):
            for layer in self.layers:
                inp = layer.forward(inp)
            return inp[0] if len(inp) == 1 else inp

        def backward(self):
            for layer in self.layers:
                layer.backward()

In [503]:
x = [[2.0,3.0,-1.0],[3.0,-1.0,0.5],[0.5,1.0,1.0],[1,0,1.0,1.0]]
y = [1.0,0,1.0,0]
mlp = MLP(3,[4,4,4,1])

In [507]:
y_pred = [mlp.forward(xi) for xi in x]
print(y_pred)

[Value(data=0.9907611612480132), Value(data=0.02825074347833003), Value(data=0.989246571430521), Value(data=0.028678574944252633)]


In [508]:
loss = sum((y-yp)**2 for y,yp in zip(y,y_pred))
print(loss)

Value(data=0.001821557535395623)


In [500]:
loss.backward()
mlp.layers[0].ns[0].w[0]
mlp.layers[0].ns[0].w[0].grad

-1.2442827104000794e-07

In [480]:
mlp.backward()

In [506]:
for _ in range(1000):
    y_pred = [mlp.forward(xi) for xi in x]
    loss = sum((y-yp)**2 for y,yp in zip(y,y_pred))
    loss.backward()
    mlp.backward()